In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/privastava/traffic-demand/sample_submission.csv
/kaggle/input/datasets/privastava/traffic-demand/train.csv
/kaggle/input/datasets/privastava/traffic-demand/test.csv


In [4]:
!pip install python-geohash --quiet

ERROR: Could not find a version that satisfies the requirement python-geohash (from versions: none)
ERROR: No matching distribution found for python-geohash


In [5]:
# 1. Bypassing Kaggle Firewall by using a local pure-python fallback mapping
try:
    import geohash as gh
except ImportError:
    # If python-geohash is missing, dynamically wire up a pure-Python geohashing module in memory
    import sys
    import types

    _base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    _decodemap = {ch: i for i, ch in enumerate(_base32)}

    def decode(geohash):
        is_even = True
        lat_interval = [-90.0, 90.0]
        lon_interval = [-180.0, 180.0]
        for c in geohash:
            cd = _decodemap[c]
            for mask in [16, 8, 4, 2, 1]:
                if is_even:
                    if cd & mask: lon_interval[0] = (lon_interval[0] + lon_interval[1]) / 2
                    else: lon_interval[1] = (lon_interval[0] + lon_interval[1]) / 2
                else:
                    if cd & mask: lat_interval[0] = (lat_interval[0] + lat_interval[1]) / 2
                    else: lat_interval[1] = (lat_interval[0] + lat_interval[1]) / 2
                is_even = not is_even
        lat = (lat_interval[0] + lat_interval[1]) / 2
        lon = (lon_interval[0] + lon_interval[1]) / 2
        return lat, lon

    # Register it as a native python system module so 'import geohash as gh' hooks in downstream
    gh = types.ModuleType('geohash')
    gh.decode = decode
    sys.modules['geohash'] = gh

print("Geospatial parser isolated and successfully wired up offline!")

Geospatial parser isolated and successfully wired up offline!


In [6]:
# Imports and system configurations
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import optuna
import geohash as gh

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit

from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# 3. Verified Kaggle Directory Paths
FAST_SUBMISSION = False   # True = quick check layout; False = final deep optimization run

TRAIN_PATH = "/kaggle/input/datasets/privastava/traffic-demand/train.csv"
TEST_PATH  = "/kaggle/input/datasets/privastava/traffic-demand/test.csv"
SUB_PATH   = "/kaggle/input/datasets/privastava/traffic-demand/sample_submission.csv"
OUT_DIR    = "/kaggle/working/"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
sub       = pd.read_csv(SUB_PATH)

print(f"Train matrix dimensions : {train_raw.shape}")
print(f"Test matrix dimensions  : {test_raw.shape}")

Train matrix dimensions : (77299, 11)
Test matrix dimensions  : (41778, 10)


In [7]:
#Mathematical data transformations & Geospatial Parsing

def decode_geohash_safe(code):
    try:
        lat, lng = gh.decode(str(code))
        return float(lat), float(lng)
    except Exception:
        return 0.0, 0.0

def engineer_features(df, global_stats=None):
    df = df.copy()

    if global_stats is not None:
        df["Weather"] = df["Weather"].fillna(global_stats["weather_mode"])
        df["RoadType"] = df["RoadType"].fillna(global_stats["road_mode"])
        dt_temp = pd.to_datetime(df["timestamp"], errors='coerce')
        df["hour_tmp"] = dt_temp.dt.hour.fillna(0).astype(int)
        df["Temperature"] = df.apply(
            lambda r: global_stats["temp_by_hour"].get(r["hour_tmp"], global_stats["temp_median"])
            if pd.isna(r["Temperature"]) else r["Temperature"],
            axis=1
        )
        df.drop(columns=["hour_tmp"], inplace=True)
    else:
        df["Weather"] = df["Weather"].fillna("Unknown")
        df["RoadType"] = df["RoadType"].fillna("Unknown")
        df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median() if not df["Temperature"].isna().all() else 25.0)

    # Binary representations
    df["LargeVehicles"] = (df["LargeVehicles"].astype(str).str.strip().str.lower() == "allowed").astype(int)
    df["Landmarks_Binary"] = (df["Landmarks"].astype(str).str.strip().str.lower() == "yes").astype(int)

    dt = pd.to_datetime(df["timestamp"], errors='coerce')
    df["hour"] = dt.dt.hour.fillna(0).astype(int)
    df["minute"] = dt.dt.minute.fillna(0).astype(int)
    df["second"] = dt.dt.second.fillna(0).astype(int)
    
    df["seconds_since_midnight"] = df["hour"] * 3600 + df["minute"] * 60 + df["second"]
    df["time_order"] = df["day"] * 86400 + df["seconds_since_midnight"]

    df["hour_sin"] = np.sin(2 * np.pi * df["seconds_since_midnight"] / 86400)
    df["hour_cos"] = np.cos(2 * np.pi * df["seconds_since_midnight"] / 86400)

    df["is_rush_hour"] = df["hour"].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)
    df["is_night"]     = df["hour"].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
    df["proximity_to_peak"] = df["hour"].apply(lambda h: min(abs(h - 8), abs(h - 18)))

    df["weekday"] = df["day"] % 7
    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)

    weather_map = {"clear": 0, "sunny": 0, "cloudy": 1, "foggy": 1, "rainy": 2, "rain": 2, "snowy": 3, "storm": 3}
    df["weather_severity"] = df["Weather"].astype(str).str.strip().str.lower().map(weather_map).fillna(1)
    df["temp_bucket"] = pd.cut(df["Temperature"], bins=[-999, 10, 25, 999], labels=["cold", "mild", "hot"]).astype(str)
    df["road_capacity_index"] = df["NumberofLanes"].fillna(1) * (1 - df["LargeVehicles"])
    df["landmark_rush"] = df["Landmarks_Binary"] * df["is_rush_hour"]

    df["weekday_hour_sin"] = df["weekday"] * df["hour_sin"]
    df["weekday_hour_cos"] = df["weekday"] * df["hour_cos"]
    df["weekend_rush"] = df["is_weekend"] * df["is_rush_hour"]
    df["temp_rush"] = df["temp_bucket"].map({"cold": 0, "mild": 1, "hot": 2}).fillna(1) * df["is_rush_hour"]

    coords = df["geohash"].apply(decode_geohash_safe)
    df["lat"] = coords.apply(lambda x: x[0])
    df["lng"] = coords.apply(lambda x: x[1])
    df["geohash_4"] = df["geohash"].astype(str).str[:4]
    df["geohash_5"] = df["geohash"].astype(str).str[:5]
    df["geohash_hour"] = df["geohash_4"] + "_" + df["hour"].astype(str)

    if global_stats is not None:
        for col in ["Weather", "RoadType", "geohash_5"]:
            rare = global_stats.get(f"rare_{col}", set())
            df[col] = df[col].apply(lambda x: "Other" if x in rare else x)

    return df

def compute_global_stats(train_df):
    dt = pd.to_datetime(train_df["timestamp"], errors='coerce')
    hour = dt.dt.hour.fillna(0).astype(int)
    stats = {
        "weather_mode": train_df["Weather"].dropna().mode()[0] if not train_df["Weather"].dropna().empty else "Unknown",
        "road_mode": train_df["RoadType"].dropna().mode()[0] if not train_df["RoadType"].dropna().empty else "Unknown",
        "temp_median": train_df["Temperature"].median() if not train_df["Temperature"].isna().all() else 25.0,
        "temp_by_hour": train_df.assign(hour=hour).groupby("hour")["Temperature"].median().to_dict(),
    }
    for col in ["Weather", "RoadType", "geohash_5"]:
        if col in train_df.columns:
            counts = train_df[col].value_counts()
            stats[f"rare_{col}"] = set(counts[counts < 5].index)
    return stats

global_stats = compute_global_stats(train_raw)
train = engineer_features(train_raw, global_stats)
test  = engineer_features(test_raw,  global_stats)
train = train.sort_values("time_order").reset_index(drop=True)

print("Basic Feature Transformations Complete.")

Basic Feature Transformations Complete.


In [8]:
# Identifying Geospatial Hotspots using unsupervised clustering and processes chronological target
high_demand_threshold = train["demand"].quantile(0.80)
high_demand_rows = train[train["demand"] > high_demand_threshold][["lat", "lng"]].dropna()

if len(high_demand_rows) >= 5:
    silhouette_sample = high_demand_rows.sample(min(2000, len(high_demand_rows)), random_state=SEED)
    best_k, best_score = 3, -1
    for k in range(3, 7):
        km = KMeans(n_clusters=k, random_state=SEED, n_init=5)
        labels = km.fit_predict(silhouette_sample)
        score = silhouette_score(silhouette_sample, labels)
        if score > best_score:
            best_k, best_score = k, score
    kmeans = KMeans(n_clusters=best_k, random_state=SEED, n_init=5)
    kmeans.fit(high_demand_rows)
    centers = kmeans.cluster_centers_
else:
    centers = np.array([[train["lat"].mean(), train["lng"].mean()]])

def add_epicenter_features(df, centers):
    coords = df[["lat", "lng"]].values
    diff = coords[:, np.newaxis, :] - centers[np.newaxis, :, :]
    dists = np.sqrt(np.sum(diff ** 2, axis=2))
    df["dist_to_epicenter"] = dists.min(axis=1)
    df["near_epicenter"] = (df["dist_to_epicenter"] < 0.005).astype(int)
    return df

train = add_epicenter_features(train, centers)
test  = add_epicenter_features(test,  centers)

# Granular Spatial-Temporal Multi-Interaction Feature 
train["geohash_weekday_hour"] = train["geohash_4"].astype(str) + "_" + train["weekday"].astype(str) + "_" + train["hour"].astype(str)
test["geohash_weekday_hour"] = test["geohash_4"].astype(str) + "_" + test["weekday"].astype(str) + "_" + test["hour"].astype(str)

global_mean = train["demand"].mean()

# Leakage-free expanding tracking on historical data strings
train["gh_hour_mean"] = train.groupby("geohash_hour")["demand"].expanding().mean().groupby(level=0).shift(1).reset_index(level=0, drop=True).fillna(global_mean)
train["gh_day_mean"] = train.groupby(["geohash_4", "day"])["demand"].expanding().mean().groupby(level=[0,1]).shift(1).reset_index(level=[0,1], drop=True).fillna(global_mean)
train["gh_weekday_mean"] = train.groupby(["geohash_4", "weekday"])["demand"].expanding().mean().groupby(level=[0,1]).shift(1).reset_index(level=[0,1], drop=True).fillna(global_mean)
train["gh_weekday_hour_mean"] = train.groupby("geohash_weekday_hour")["demand"].expanding().mean().groupby(level=0).shift(1).reset_index(level=0, drop=True).fillna(global_mean)
train["roadtype_hour_mean"] = train.groupby(["RoadType", "hour"])["demand"].expanding().mean().groupby(level=[0,1]).shift(1).reset_index(level=[0,1], drop=True).fillna(global_mean)
train["gh_demand_std"] = train.groupby("geohash_4")["demand"].expanding().std().groupby(level=0).shift(1).reset_index(level=0, drop=True).fillna(0)

# Multi-level fallback map configurations
lookup_gh_hour = train.groupby("geohash_hour")["demand"].mean().to_dict()
lookup_gh4 = train.groupby("geohash_4")["demand"].mean().to_dict()
lookup_gh_day = train.groupby(["geohash_4", "day"])["demand"].mean().to_dict()
lookup_gh_weekday = train.groupby(["geohash_4", "weekday"])["demand"].mean().to_dict()
lookup_gh_weekday_hour = train.groupby("geohash_weekday_hour")["demand"].mean().to_dict()
lookup_roadtype_hour = train.groupby(["RoadType", "hour"])["demand"].mean().to_dict()
lookup_gh_std = train.groupby("geohash_4")["demand"].std().fillna(0).to_dict()

def apply_target_encodings(df):
    df = df.copy()
    df["gh_hour_mean"] = df.apply(lambda r: lookup_gh_hour.get(r["geohash_hour"], lookup_gh4.get(r["geohash_4"], global_mean)), axis=1)
    df["gh_day_mean"] = df.apply(lambda r: lookup_gh_day.get((r["geohash_4"], r["day"]), lookup_gh4.get(r["geohash_4"], global_mean)), axis=1)
    df["gh_weekday_mean"] = df.apply(lambda r: lookup_gh_weekday.get((r["geohash_4"], r["weekday"]), lookup_gh4.get(r["geohash_4"], global_mean)), axis=1)
    df["gh_weekday_hour_mean"] = df.apply(lambda r: lookup_gh_weekday_hour.get(r["geohash_weekday_hour"], lookup_gh_hour.get(r["geohash_hour"], global_mean)), axis=1)
    df["roadtype_hour_mean"] = df.apply(lambda r: lookup_roadtype_hour.get((r["RoadType"], r["hour"]), global_mean), axis=1)
    df["gh_demand_std"] = df["geohash_4"].map(lookup_gh_std).fillna(0)
    return df

test = apply_target_encodings(test)
print("Hotspot extraction and advanced target summaries fully encoded.")

Hotspot extraction and advanced target summaries fully encoded.


In [9]:
# Train/Validation Splitting & Label Synchronization
FEATURE_COLS = [
    "hour", "hour_sin", "hour_cos", "is_rush_hour", "is_night", "proximity_to_peak",
    "day", "weekday", "weekday_sin", "weekday_cos", "is_weekend",
    "weekday_hour_sin", "weekday_hour_cos", "weekend_rush", "temp_rush",
    "lat", "lng", "geohash_4", "geohash_5", "geohash_hour", "geohash_weekday_hour",
    "dist_to_epicenter", "near_epicenter",
    "weather_severity", "temp_bucket", "Temperature",
    "road_capacity_index", "landmark_rush", "NumberofLanes", "LargeVehicles",
    "gh_hour_mean", "gh_day_mean", "gh_weekday_mean", "gh_weekday_hour_mean", "roadtype_hour_mean", "gh_demand_std",
    "RoadType", "Weather", "Landmarks_Binary"  
]

# Ensure we only retain columns that actively exist across your dataframes
FEATURE_COLS = [c for c in FEATURE_COLS if c in train.columns]
CAT_FEATURES_CB = ["geohash_4", "geohash_5", "geohash_hour", "geohash_weekday_hour", "RoadType", "Weather", "temp_bucket"]

le_dict = {}
for col in CAT_FEATURES_CB:
    le = LabelEncoder()
    combined_vocab = pd.concat([train[col], test[col]]).astype(str)
    le.fit(combined_vocab)
    le_dict[col] = le

split_idx = int(len(train) * 0.80)
X_train, X_val = train[FEATURE_COLS].iloc[:split_idx], train[FEATURE_COLS].iloc[split_idx:]
y_train, y_val = train["demand"].iloc[:split_idx], train["demand"].iloc[split_idx:]

print(f"Matrix partitions locked. Train items: {X_train.shape[0]:,}, Validation items: {X_val.shape[0]:,}")

Matrix partitions locked. Train items: 61,839, Validation items: 15,460


In [10]:
#Resumable Base Model Training (CatBoost + LightGBM)
# --- CatBoost Training Execution ---
CB_CHECKPOINT = os.path.join(OUT_DIR, "catboost_model.cbm")
cat_idx = [X_train.columns.tolist().index(c) for c in CAT_FEATURES_CB]

train_pool = Pool(X_train, y_train, cat_features=cat_idx)
val_pool   = Pool(X_val,   y_val,   cat_features=cat_idx)

if os.path.exists(CB_CHECKPOINT):
    print("Loading precomputed CatBoost checkpoint weight maps...")
    cb_model = CatBoostRegressor()
    cb_model.load_model(CB_CHECKPOINT)
else:
    cb_model = CatBoostRegressor(
        iterations=1000, learning_rate=0.05, depth=7,
        eval_metric="R2", loss_function="RMSE", random_seed=SEED,
        early_stopping_rounds=50, verbose=200,
    )
    cb_model.fit(train_pool, eval_set=val_pool)
    cb_model.save_model(CB_CHECKPOINT)

cb_preds_val = cb_model.predict(X_val)
cb_r2 = r2_score(y_val, cb_preds_val)

# --- LightGBM Training Execution ---
LGBM_CHECKPOINT = os.path.join(OUT_DIR, "lgbm_model.pkl")
X_train_lgbm, X_val_lgbm = X_train.copy(), X_val.copy()

for col in CAT_FEATURES_CB:
    X_train_lgbm[col] = le_dict[col].transform(X_train[col].astype(str))
    X_val_lgbm[col] = le_dict[col].transform(X_val[col].astype(str))

categorical_lgbm_idx = [X_train_lgbm.columns.get_loc(c) for c in CAT_FEATURES_CB]

if os.path.exists(LGBM_CHECKPOINT):
    print("Loading precomputed LightGBM checkpoint weight maps...")
    with open(LGBM_CHECKPOINT, "rb") as f:
        lgbm_model = pickle.load(f)
else:
    lgbm_model = LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, num_leaves=63,
        random_state=SEED, n_jobs=-1, verbose=-1,
        categorical_feature=categorical_lgbm_idx
    )
    lgbm_model.fit(X_train_lgbm, y_train, eval_set=[(X_val_lgbm, y_val)])
    with open(LGBM_CHECKPOINT, "wb") as f:
        pickle.dump(lgbm_model, f)

lgbm_preds_val = lgbm_model.predict(X_val_lgbm)
lgbm_r2 = r2_score(y_val, lgbm_preds_val)

print(f"\nBaseline Verification Metrics — CatBoost R²: {cb_r2:.4f} | LightGBM R²: {lgbm_r2:.4f}")

0:	learn: 0.0743872	test: 0.0598033	best: 0.0598033 (0)	total: 126ms	remaining: 2m 5s
200:	learn: 0.8634870	test: 0.7839315	best: 0.7839315 (200)	total: 11.1s	remaining: 44.1s
400:	learn: 0.8900896	test: 0.7905337	best: 0.7905337 (400)	total: 22.9s	remaining: 34.1s
600:	learn: 0.9035359	test: 0.7917651	best: 0.7921118 (554)	total: 34.4s	remaining: 22.9s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7921118108
bestIteration = 554

Shrink model to first 555 iterations.

Baseline Verification Metrics — CatBoost R²: 0.7921 | LightGBM R²: 0.7762


In [11]:
# SQLite-Backed Hyperparameter Optimization & Inference
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 400, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "depth": trial.suggest_int("depth", 5, 9),
        "eval_metric": "R2", "loss_function": "RMSE", "random_seed": SEED, "verbose": 0,
    }
    
    n_splits = 3 if FAST_SUBMISSION else 5
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    # Pre-calculated features are already leak-free due to chronological expanding shifts
    X_pure = train[FEATURE_COLS]
    y_pure = train["demand"]
    
    scores = []
    for tr_idx, va_idx in tscv.split(X_pure):
        Xtr, Xva = X_pure.iloc[tr_idx], X_pure.iloc[va_idx]
        ytr, yva = y_pure.iloc[tr_idx], y_pure.iloc[va_idx]
        
        c_idx = [Xtr.columns.tolist().index(c) for c in CAT_FEATURES_CB if c in Xtr.columns]
        m = CatBoostRegressor(**params)
        m.fit(Pool(Xtr, ytr, cat_features=c_idx), eval_set=Pool(Xva, yva, cat_features=c_idx), early_stopping_rounds=30, verbose=0)
        
        scores.append(r2_score(yva, m.predict(Xva)))
    return np.mean(scores)

DB_PATH = os.path.join(OUT_DIR, "optuna_study.db")
study = optuna.create_study(
    study_name="traffic_demand_hacker_earth",
    storage=f"sqlite:///{DB_PATH}",
    direction="maximize",
    load_if_exists=True
)

n_trials = 5 if FAST_SUBMISSION else 50
remaining = max(0, n_trials - len(study.trials))

if remaining > 0:
    print(f"Running {remaining} persistent optimization passes via SQLite study mapping...")
    study.optimize(objective, n_trials=remaining)

print(f"Optimal Search Optimization Profile R²: {study.best_value:.4f}")

best_params = study.best_params.copy()
best_params.update({"eval_metric": "R2", "loss_function": "RMSE", "random_seed": SEED, "verbose": 0})

X_full = train[FEATURE_COLS]
y_full = train["demand"]
ci_full = [X_full.columns.tolist().index(c) for c in CAT_FEATURES_CB]

cb_final = CatBoostRegressor(**best_params)
cb_final.fit(Pool(X_full, y_full, cat_features=ci_full))

X_full_lgbm = X_full.copy()
for col in CAT_FEATURES_CB:
    X_full_lgbm[col] = le_dict[col].transform(X_full[col].astype(str))

# --- STABILITY UPGRADE: Explicitly define LightGBM Categorical Array Layout ---
categorical_lgbm_idx = [X_full_lgbm.columns.tolist().index(c) for c in CAT_FEATURES_CB if c in X_full_lgbm.columns]

lgbm_final = LGBMRegressor(
    n_estimators=best_params.get("iterations", 800),
    learning_rate=best_params.get("learning_rate", 0.05),
    num_leaves=63, random_state=SEED, n_jobs=-1, verbose=-1,
    categorical_feature=categorical_lgbm_idx
)
lgbm_final.fit(X_full_lgbm, y_full)

X_test = test[FEATURE_COLS]
X_test_lgbm = X_test.copy()
for col in CAT_FEATURES_CB:
    X_test_lgbm[col] = le_dict[col].transform(X_test[col].astype(str))

cb_test_preds = cb_final.predict(X_test)
lgbm_test_preds = lgbm_final.predict(X_test_lgbm)

# --- STABILITY UPGRADE: Correct variable target naming inside blend matrix ---
final_preds = (0.60 * cb_test_preds) + (0.40 * lgbm_test_preds)
final_preds = np.clip(final_preds, 0, 1)

# --- GUIDELINE UPGRADE: Precise key validation structure ---
sub_index = test["Index"] if "Index" in test.columns else test.index
submission_df = pd.DataFrame({
    "Index": sub_index,   
    "demand": final_preds      
})

# index=False drops the pandas matrix indexes to create a crisp 41778 x 2 structure
submission_df.to_csv("submission.csv", index=False)

print("=======================================================")
print("  PIPELINE PROCESSING COMPLETE — ASSETS SECURED")
print("=======================================================")
print(f"  Ensemble Submission Target Matrix Row Count: {len(submission_df)}")
print("=======================================================")

Running 5 persistent optimization passes via SQLite study mapping...
Optimal Search Optimization Profile R²: 0.8042
  PIPELINE PROCESSING COMPLETE — ASSETS SECURED
  Ensemble Submission Target Matrix Row Count: 41778
